In [ ]:
import sys
sys.path.insert(1, '../utils/')

In [ ]:
import datacube
import time

from IPython.display import HTML, display

from utils.deafrica_plotting import rgb
from utils.nostra_dc import get_product_bbox
from utils.nostra_mapping import bbox_to_polygon, display_crosshair, get_utm_epsg_code, MapHandler
from utils.nostra_tools import style_output_cells

In [ ]:
dc = datacube.Datacube(app="Test_MinIO_indexation")

In [ ]:
# Check if default bbox is contained within the datacube
# and allow user to draw bbox if not.

product = 'ls89_c2l2_sp_local'

product_bbox = get_product_bbox(dc, product, split_size=10, stability_threshold=4)

In [ ]:
# Create an instance of MapHandler
map_handler = MapHandler()
m, drc = map_handler.create_map(vect=[bbox_to_polygon(product_bbox)],
                               draw_rect=True)
display(m)

# append crosshair
time.sleep(2)  # make sure m is fully displayed
display_crosshair()

In [ ]:
# Warn in case of full AoI
aoi_poly = map_handler.aoi_tupple

if aoi_poly is None:
    style_output_cells('salmon', border_color='red', border_width='2px')
    print('The area of interest polygon (aoi_poly) has not been created. Please draw it in the previous cell.')
else:
    # A polygon was drawn
    style_output_cells()
    print('Custom area of interest polygon has been created.')

In [ ]:
# get EPSG code for the center of the AoI

epsg_code = get_utm_epsg_code((aoi_poly[1] + aoi_poly[3]) / 2, (aoi_poly[0] + aoi_poly[2]) / 2)

In [ ]:
dss = dc.find_datasets(product=product,
                       x=(aoi_poly[0], aoi_poly[2]),
                       y=(aoi_poly[1], aoi_poly[3]))
times = [ds.time.begin for ds in dss]
start_date = min(times)
end_date = max(times)

In [ ]:
lazy_ds = dc.load(product=product,
                  measurements=['red', 'green', 'blue'],
                  output_crs=f"EPSG:{epsg_code}",
                  y=(aoi_poly[1], aoi_poly[3]),
                  x=(aoi_poly[0], aoi_poly[2]),
                  time=(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')),
                  resolution=30,
                  group_by="solar_day",
                  dask_chunks={},
                  )

In [ ]:
# customized chunking is less buggy when performed after loading !!!
lazy_ds = lazy_ds.chunk({"x": 512, "y": 512, "time": 1})
print(lazy_ds)

In [ ]:
rgb(lazy_ds, col="time", col_wrap=4, robust=True)